In [ ]:
#Zstd with delta encoding

#HIV-1

import os

import sys

import time

import hashlib

import zstandard as zstd

from pathlib import Path



# -----------------------------------------------------------------------------

# CONFIGURATION

# -----------------------------------------------------------------------------

INPUT_FILE_PATH = Path(r"C:\Users\swapn\OneDrive\Desktop\Bio_Project\sequences.fasta")

OUTPUT_DIR = Path(r"C:\Users\swapn\OneDrive\Desktop\Bio_Project\sequences_zstd_results")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



COMPRESSED_FILE = OUTPUT_DIR / "sequences_compressed.zst"

DECOMPRESSED_FILE = OUTPUT_DIR / "sequences_decompressed.fasta"



# -----------------------------------------------------------------------------

# HELPERS

# -----------------------------------------------------------------------------

def sha256_hash(data: bytes) -> str:

    h = hashlib.sha256()

    h.update(data)

    return h.hexdigest()



def safe_speed(size_bytes: int, elapsed: float) -> float:

    return (size_bytes / elapsed) if elapsed > 0 else 0.0



# -----------------------------------------------------------------------------

# 1) SEQUENCE ANALYSIS

# -----------------------------------------------------------------------------

print("SEQUENCE ANALYSIS")

print("-" * 40)



# Read all sequences into memory

with INPUT_FILE_PATH.open("r") as f:

    lines = [l.strip() for l in f if not l.startswith(">")]

sequence = "".join(lines)

length = len(sequence)



# Composition counts

counts = {

    "A": sequence.count("A"),

    "T": sequence.count("T"),

    "G": sequence.count("G"),

    "C": sequence.count("C"),

}



print(f"Number of sequences: 1")

print(f"Total length: {length:,} bases")

print(f"Average length: {length:,} bases")

print(f"Min length: {length:,} bases")

print(f"Max length: {length:,} bases\n")



print("Base composition:")

for base, cnt in counts.items():

    pct = cnt / length * 100 if length > 0 else 0

    print(f"  {base}: {cnt:,} ({pct:.1f}%)")

print()



# Prepare raw bytes and hash

raw_bytes = sequence.encode("utf-8")

original_hash = sha256_hash(raw_bytes)



# -----------------------------------------------------------------------------

# 2) ZSTD COMPRESSION

# -----------------------------------------------------------------------------

print("ZSTD COMPRESSION PHASE")

print("-" * 40)



orig_size = len(raw_bytes)

print(f"Original size: {orig_size:,} bytes")

print(f"Original hash: {original_hash[:16]}...")



compressor = zstd.ZstdCompressor()

start = time.time()

compressed = compressor.compress(raw_bytes)

elapsed_c = time.time() - start



with COMPRESSED_FILE.open("wb") as out_f:

    out_f.write(compressed)



comp_size = len(compressed)

ratio = orig_size / comp_size if comp_size > 0 else 0

space_saved = (1 - comp_size / orig_size) * 100 if orig_size > 0 else 0

speed_c = safe_speed(orig_size, elapsed_c) / (1024 * 1024)  # MB/s



print("Compressing...")

print("Compression completed")

print(f"Compressed size: {comp_size:,} bytes")

print(f"Compression ratio: {ratio:.2f}:1")

print(f"Space saved: {space_saved:.1f}%")

print(f"Compression time: {elapsed_c:.2f} seconds")

print(f"Compression speed: {speed_c:.2f} MB/s")

print(f"Output file: {COMPRESSED_FILE}\n")



# -----------------------------------------------------------------------------

# 3) ZSTD DECOMPRESSION

# -----------------------------------------------------------------------------

print("ZSTD DECOMPRESSION PHASE")

print("-" * 40)



try:

    compressed_data = COMPRESSED_FILE.read_bytes()

    dctx = zstd.ZstdDecompressor()



    print("Reading compressed file...")

    start = time.time()

    decompressed = dctx.decompress(compressed_data)

    elapsed_d = time.time() - start



    with DECOMPRESSED_FILE.open("wb") as out_f:

        out_f.write(decompressed)



    dec_size = len(decompressed)

    speed_d = safe_speed(dec_size, elapsed_d) / (1024 * 1024)  # MB/s



    print("Decompressing...")

    print("Decompression completed")

    print(f"Decompressed size: {dec_size:,} bytes")

    print(f"Decompression time: {elapsed_d:.2f} seconds")

    print(f"Decompression speed: {speed_d:.2f} MB/s")

    print(f"Output file: {DECOMPRESSED_FILE}\n")



    decomp_success = True



except Exception as e:

    print("Decompression failed:", e)

    decomp_success = False



# -----------------------------------------------------------------------------

# 4) INTEGRITY VERIFICATION

# -----------------------------------------------------------------------------

print("INTEGRITY VERIFICATION")

print("-" * 40)



if decomp_success:

    new_bytes = DECOMPRESSED_FILE.read_bytes()

    new_hash = sha256_hash(new_bytes)

    if new_hash == original_hash:

        print("Verification: 100% verified")

    else:

        print("Verification: Hash mismatch")

else:

    print("Cannot verify - decompression failed")


Namah Shivaya


In [ ]:
#BWT+ZSTD

#HIV-1

import os, sys, time, hashlib, zstandard as zstd

from pathlib import Path

from typing import List, Tuple, Dict



SEQ_DELIMITER = ">SEQ<"



class BWTProcessor:

    @staticmethod

    def bwt_encode(text: str) -> Tuple[str, int]:

        text = text + "$"

        rotations = sorted(text[i:] + text[:i] for i in range(len(text)))

        full_bwt = "".join(row[-1] for row in rotations)

        original_index = rotations.index(text)

        return full_bwt, original_index



    @staticmethod

    def bwt_decode(full_bwt: str, original_index: int) -> str:

        bwt = full_bwt

        n = len(bwt)

        freq = {}

        for ch in bwt:

            freq[ch] = freq.get(ch, 0) + 1

        chars = sorted(freq)

        C = {}

        tot = 0

        for ch in chars:

            C[ch] = tot

            tot += freq[ch]

        Occ = []

        counts = {ch: 0 for ch in chars}

        for ch in bwt:

            Occ.append(counts.copy())

            counts[ch] += 1

        idx = original_index

        res = []

        for _ in range(n):

            ch = bwt[idx]

            res.append(ch)

            idx = C[ch] + Occ[idx][ch]

        text_with_marker = "".join(reversed(res))

        return text_with_marker.rstrip("$")



def read_fasta_with_headers(path: Path) -> Tuple[List[str], List[str]]:

    headers, seqs, curr_seq = [], [], []

    curr_hdr = None

    with open(path, 'r') as f:

        for line in f:

            line = line.rstrip("\n")

            if line.startswith(">"):

                if curr_hdr is not None:

                    headers.append(curr_hdr)

                    seqs.append("".join(curr_seq).upper())

                curr_hdr = line

                curr_seq = []

            else:

                curr_seq.append(line)

        if curr_hdr is not None:

            headers.append(curr_hdr)

            seqs.append("".join(curr_seq).upper())

    return headers, seqs



def write_fasta(path: Path, headers: List[str], seqs: List[str]):

    with open(path, 'w') as f:

        for h, s in zip(headers, seqs):

            f.write(f"{h}\n{s}\n")



def prepare_bwt_data(seqs: List[str]) -> Dict:

    combined = SEQ_DELIMITER.join(seqs)

    bwt, idx = BWTProcessor.bwt_encode(combined)

    orig_bytes = combined.encode('utf-8')

    return {

        "combined": combined,

        "orig_len": len(orig_bytes),

        "orig_hash": hashlib.sha256(orig_bytes).hexdigest(),

        "bwt": bwt,

        "index": idx

    }



def compress_zstd(bd: Dict, path: Path) -> Dict:

    out_dir = path.parent / (path.stem + "_bwt_zstd")

    out_dir.mkdir(exist_ok=True)

    payload = f"{bd['index']}{SEQ_DELIMITER}{bd['bwt']}".encode('utf-8')

    start = time.time()

    comp = zstd.ZstdCompressor(level=22).compress(payload)

    ctime = time.time() - start

    out_file = out_dir / (path.stem + ".zst")

    with open(out_file, 'wb') as f:

        f.write(comp)

    return {

        "comp": comp,

        "len": len(comp),

        "time": ctime,

        "file": out_file,

        "out_dir": out_dir

    }



def decompress_and_reverse(bd: Dict, cd: Dict) -> Dict:

    with open(cd["file"], 'rb') as f:

        comp = f.read()

    start = time.time()

    dec = zstd.ZstdDecompressor().decompress(comp).decode('utf-8')

    dtime = time.time() - start

    idx_str, bwt = dec.split(SEQ_DELIMITER, 1)

    recon = BWTProcessor.bwt_decode(bwt, int(idx_str))

    recon_bytes = recon.encode('utf-8')

    return {

        "recon": recon,

        "len": len(recon_bytes),

        "hash": hashlib.sha256(recon_bytes).hexdigest(),

        "time": dtime

    }



def verify(bd: Dict, rd: Dict):

    print("\n=== Integrity Verification ===")

    size_match = bd["orig_len"] == rd["len"]

    hash_match = bd["orig_hash"] == rd["hash"]

    print(f"Size Match : {'PASS' if size_match else 'FAIL'}")

    print(f"Hash Match : {'PASS' if hash_match else 'FAIL'}")

    if not (size_match and hash_match):

        print("Data Integrity : FAIL")

        sys.exit(1)

    print("Data Integrity : PASS")



def run_pipeline(fasta: str):

    p = Path(fasta)

    print("Reading input...")

    headers, seqs = read_fasta_with_headers(p)



    print("Applying BWT...")

    bd = prepare_bwt_data(seqs)



    print("Compressing with Zstandard...")

    cd = compress_zstd(bd, p)

    cr = bd["orig_len"] / cd["len"]



    print("Decompressing and Reversing BWT...")

    rd = decompress_and_reverse(bd, cd)



    verify(bd, rd)



    out_fasta = cd["out_dir"] / (p.stem + "_decoded.fasta")

    write_fasta(out_fasta, headers, rd["recon"].split(SEQ_DELIMITER))



    print("\n=== Compression Report ===")

    print(f"Original size     : {bd['orig_len']} bytes")

    print(f"Compressed size   : {cd['len']} bytes")

    print(f"Compression ratio : {cr:.2f}:1")

    print(f"Compression time  : {cd['time']:.3f} sec")

    print(f"Decompression time: {rd['time']:.3f} sec")

    print(f"Decompressed file : {out_fasta}")

    print(f"Compressed file   : {cd['file']}")

    print("\nPipeline complete.")



if _name_ == "_main_":

    run_pipeline(r"C:\Users\swapn\OneDrive\Desktop\Bio_Project\sequences.fasta")

In [ ]:
#RLE+ZSTD

#HIV-1

# Cell 1: Setup & Imports

import os

import sys

import time

import hashlib

import zstandard as zstd

from pathlib import Path

from typing import List, Dict, Any

import pandas as pd

from tqdm import tqdm

import matplotlib.pyplot as plt



# Configuration

INPUT_FILE_PATH = r"C:\Users\swapn\OneDrive\Desktop\Bio_Project\sequences.fasta"

print("RLE + ZSTD Viral Genome Compression Pipeline")

print("=" * 60)

print(f"Input file: {INPUT_FILE_PATH}")



# Cell 2: File Reading & Analysis

def read_genome_file(file_path: Path) -> List[str]:

    """Read genome sequences from FASTA file"""

    sequences = []

    current_sequence = ""

    with open(file_path, 'r') as f:

        for line in tqdm(f, desc="Reading FASTA"):

            line = line.strip()

            if line.startswith('>'):

                if current_sequence:

                    sequences.append(current_sequence.upper())

                    current_sequence = ""

            else:

                current_sequence += line

    if current_sequence:

        sequences.append(current_sequence.upper())

    return sequences



def analyze_sequences(sequences: List[str]) -> Dict[str, Any]:

    """Analyze sequence characteristics"""

    total_length = sum(len(seq) for seq in sequences)

    avg_length = total_length / len(sequences) if sequences else 0

    # Base composition

    all_bases = ''.join(sequences)

    base_counts = {base: all_bases.count(base) for base in 'ATGC'}

    return {

        'sequence_count': len(sequences),

        'total_length': total_length,

        'average_length': avg_length,

        'min_length': min(len(seq) for seq in sequences) if sequences else 0,

        'max_length': max(len(seq) for seq in sequences) if sequences else 0,

        'base_composition': base_counts

    }



# Read and analyze sequences

if not os.path.exists(INPUT_FILE_PATH):

    print(f"Error: File '{INPUT_FILE_PATH}' not found!")

    sys.exit(1)



sequences = read_genome_file(Path(INPUT_FILE_PATH))

analysis = analyze_sequences(sequences)

print("\nSEQUENCE ANALYSIS")

print("-" * 40)

print(f"Number of sequences: {analysis['sequence_count']}")

print(f"Total length: {analysis['total_length']} bases")

print("\nBase composition:")

for base, count in analysis['base_composition'].items():

    percentage = (count / analysis['total_length']) * 100

    print(f"  {base}: {count} ({percentage:.1f}%)")



# Cell 3: RLE Functions

def rle_encode(sequence: str) -> str:

    """Apply Run-Length Encoding to a single DNA sequence"""

    encoded = []

    i = 0

    while i < len(sequence):

        base = sequence[i]

        count = 1

        while i + 1 < len(sequence) and sequence[i+1] == base:

            count += 1

            i += 1

        encoded.append(f"{base}{count}")

        i += 1

    return ''.join(encoded)



def rle_decode(encoded: str) -> str:

    """Recover original sequence from RLE-encoded string"""

    decoded = []

    i = 0

    while i < len(encoded):

        base = encoded[i]

        i += 1

        num_str = ''

        while i < len(encoded) and encoded[i].isdigit():

            num_str += encoded[i]

            i += 1

        count = int(num_str) if num_str else 1

        decoded.append(base * count)

    return ''.join(decoded)



# Cell 4: Hash Utility

def calculate_hash(data: str) -> str:

    """Calculate SHA256 hash of data"""

    return hashlib.sha256(data.encode('utf-8')).hexdigest()



# Cell 5: RLE + ZSTD Compression

def compress_with_rle_zstd(sequences: List[str], file_path: Path) -> Dict[str, Any]:

    """Compress sequences using RLE + ZSTD maximum compression"""

    print("\nRLE + ZSTD COMPRESSION PHASE")

    print("-" * 40)

    # Apply RLE to each sequence

    rle_sequences = [rle_encode(seq) for seq in sequences]

    combined_data = '\n'.join(rle_sequences)

    original_size = len(combined_data.encode('utf-8'))

    original_hash = calculate_hash(combined_data)

    print(f"Original size: {original_size:,} bytes")

    print(f"Original hash: {original_hash[:16]}...")



    # Setup output directory and file

    base_name = file_path.stem

    output_dir = file_path.parent / f"{base_name}_rle_zstd_results"

    output_dir.mkdir(exist_ok=True)

    compressed_file = output_dir / f"{base_name}_compressed.zst"



    try:

        # Compress with ZSTD level 22

        print("Compressing...")

        start_time = time.perf_counter()

        compressor = zstd.ZstdCompressor(level=22)

        compressed_data = compressor.compress(combined_data.encode('utf-8'))

        compression_time = time.perf_counter() - start_time



        # Save compressed file

        with open(compressed_file, 'wb') as f:

            f.write(compressed_data)



        compressed_size = len(compressed_data)

        compression_ratio = original_size / compressed_size

        space_saved = (1 - compressed_size/original_size) * 100

        compression_speed = original_size / (1024 * 1024) / max(0.001, compression_time)



        print(f"Compression completed!")

        print(f"Compressed size: {compressed_size:,} bytes")

        print(f"Compression ratio: {compression_ratio:.2f}:1")

        print(f"Space saved: {space_saved:.1f}%")

        print(f"Compression time: {compression_time:.4f} seconds")

        print(f"Compression speed: {compression_speed:.1f} MB/s")

        print(f"Output file: {compressed_file}")



        return {

            'success': True,

            'original_size': original_size,

            'compressed_size': compressed_size,

            'compression_ratio': compression_ratio,

            'space_saved': space_saved,

            'compression_time': compression_time,

            'compression_speed': compression_speed,

            'compressed_file': str(compressed_file),

            'original_hash': original_hash,

            'original_data': combined_data,

            'original_sequences': sequences

        }

    except Exception as e:

        print(f"Compression failed: {e}")

        return {'success': False, 'error': str(e)}



# Run compression

compression_result = compress_with_rle_zstd(sequences, Path(INPUT_FILE_PATH))



# Cell 6: RLE + ZSTD Decompression

def decompress_and_verify(compression_result: Dict[str, Any]) -> Dict[str, Any]:

    """Decompress ZSTD file and verify integrity"""

    print("\nRLE + ZSTD DECOMPRESSION PHASE")

    print("-" * 40)

    if not compression_result.get('success'):

        print("Cannot decompress - compression failed")

        return {'success': False}



    compressed_file = Path(compression_result['compressed_file'])

    decompressed_file = compressed_file.parent / f"{compressed_file.stem}_decompressed.fasta"



    try:

        # Read and decompress

        print("Reading compressed file...")

        with open(compressed_file, 'rb') as f:

            compressed_data = f.read()

        print("Decompressing...")

        start_time = time.perf_counter()

        decompressor = zstd.ZstdDecompressor()

        decompressed_data = decompressor.decompress(compressed_data)

        decompression_time = time.perf_counter() - start_time



        # Decode RLE

        decompressed_string = decompressed_data.decode('utf-8')

        rle_sequences = decompressed_string.split('\n')

        recovered_sequences = [rle_decode(seq) for seq in rle_sequences]

        recovered_string = '\n'.join(recovered_sequences)



        # Save decompressed FASTA

        with open(decompressed_file, 'w') as f:

            f.write(recovered_string)



        # Metrics

        decompressed_size = len(decompressed_data)

        decompressed_hash = calculate_hash(decompressed_string)

        decompression_speed = decompressed_size / (1024 * 1024) / max(0.001, decompression_time)



        print(f"Decompression completed!")

        print(f"Decompressed size: {decompressed_size:,} bytes")

        print(f"Decompression time: {decompression_time:.4f} seconds")

        print(f"Decompression speed: {decompression_speed:.1f} MB/s")

        print(f"Decompressed hash: {decompressed_hash[:16]}...")

        print(f"Output file: {decompressed_file}")



        return {

            'success': True,

            'decompressed_size': decompressed_size,

            'decompression_time': decompression_time,

            'decompression_speed': decompression_speed,

            'decompressed_file': str(decompressed_file),

            'decompressed_hash': decompressed_hash,

            'recovered_sequences': recovered_sequences,

            'decompressed_data': recovered_string

        }

    except Exception as e:

        print(f"Decompression failed: {e}")

        return {'success': False, 'error': str(e)}



# Run decompression

decompression_result = decompress_and_verify(compression_result)



# Cell 7: Integrity Verification

def verify_integrity(compression_result: Dict[str, Any], 

                     decompression_result: Dict[str, Any]) -> Dict[str, Any]:

    """Verify complete integrity of compression-decompression cycle"""

    print("\nINTEGRITY VERIFICATION")

    print("-" * 40)

    if not compression_result.get('success') or not decompression_result.get('success'):

        print("Cannot verify - compression or decompression failed")

        return {'verified': False}



    # Get original and recovered sequences

    original_sequences = compression_result['original_sequences']

    recovered_sequences = decompression_result['recovered_sequences']



    # Size match

    size_match = len(original_sequences) == len(recovered_sequences)



    # Content match

    content_match = all(seq == recovered_sequences[i] for i, seq in enumerate(original_sequences))



    # Hash match

    original_hash = calculate_hash(''.join(original_sequences))

    recovered_hash = calculate_hash(''.join(recovered_sequences))

    hash_match = original_hash == recovered_hash



    # Sequence count match

    sequence_count_match = len(original_sequences) == len(recovered_sequences)



    print(f"Size match: {'Yes' if size_match else 'No'}")

    print(f"Hash match: {'Yes' if hash_match else 'No'}")

    print(f"Content match: {'Yes' if content_match else 'No'}")

    print(f"Sequence count match: {'Yes' if sequence_count_match else 'No'}")



    fully_verified = size_match and hash_match and content_match and sequence_count_match

    if fully_verified:

        print("\nVERIFICATION SUCCESSFUL - 100% LOSSLESS COMPRESSION CONFIRMED!")

    else:

        print("\nVERIFICATION FAILED - DATA INTEGRITY COMPROMISED!")



    return {

        'verified': fully_verified,

        'size_match': size_match,

        'hash_match': hash_match,

        'content_match': content_match,

        'sequence_count_match': sequence_count_match

    }



verification_result = verify_integrity(compression_result, decompression_result)



# Cell 8: Final Report

def generate_final_report(compression_result: Dict[str, Any],

                          decompression_result: Dict[str, Any],

                          verification_result: Dict[str, Any]):

    print("\nFINAL REPORT")

    print("-" * 40)

    print(f"Compression Ratio: {compression_result['compression_ratio']:.2f}:1")

    print(f"Space Saved: {compression_result['space_saved']:.1f}%")

    print(f"Verified: {'Yes' if verification_result['verified'] else 'No'}")



generate_final_report(compression_result, decompression_result, verification_result)



# Optional Visualization

if compression_result.get('success'):

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    sizes = [compression_result['original_size'], compression_result['compressed_size']]

    labels = ['Original', 'Compressed']

    colors = ['lightblue', 'lightcoral']

    ax1.bar(labels, sizes, color=colors)

    ax1.set_ylabel("File Size (bytes)")

    ax1.set_title("RLE + ZSTD Compression Performance")

    ax1.annotate(f"{compression_result['compression_ratio']:.1f}:1", xy=(0.5, max(sizes)/2), ha='center', fontsize=12, fontweight='bold')



    if decompression_result.get('success'):

        speeds = [compression_result['compression_speed'], decompression_result['decompression_speed']]

        speed_labels = ['Compression', 'Decompression']

        ax2.bar(speed_labels, speeds, color=['lightgreen', 'lightyellow'])

        ax2.set_ylabel("Speed (MB/s)")

        ax2.set_title("Processing Speed Comparison")

    plt.tight_layout()

    plt.show()